# 졸음 감지 모델 v2

**변경점 (v1 대비)**
- 데이터: `windows.npy`(PERCLOS 라벨) → JSON + `new_annotation_clips.csv`(수동 라벨)
- 라벨: alert / between / drowsy → **alert / drowsy (이진)**
- Split: random → **subject 단위 (data leakage 방지)**
- 모델 비교: CNN+BiLSTM / Transformer / Random Forest / Rule-based
- 증강: 적극적으로 적용

## 1. Setup

In [ ]:
import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'])

import torch
import numpy as np
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 모델: {torch.cuda.get_device_name(0)}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {DEVICE}')

from google.colab import drive
drive.mount('/content/drive')

## 2. Config

In [ ]:
class CFG:
    # ─── 경로 ───────────────────────────────────────────────────────────────
    WINDOWS_PATH  = '/content/drive/MyDrive/internship/windows_v2.npy'
    LABELS_PATH   = '/content/drive/MyDrive/internship/labels_v2.npy'
    SUBJECTS_PATH = '/content/drive/MyDrive/internship/subjects_v2.npy'
    SAVE_BASE     = '/content/drive/MyDrive/internship/runs'

    # ─── 데이터 ─────────────────────────────────────────────────────────────
    WINDOW_SIZE   = 80      # 8초 × 10fps
    FEATURE_NAMES = ['ear', 'mar', 'pitch', 'yaw', 'roll']
    N_FEATURES    = 8       # circular 변환 후 (ear, mar, sin/cos × 3)
    CLASS_NAMES   = ['alert', 'drowsy']
    N_CLASSES     = 2

    # ─── subject split ──────────────────────────────────────────────────────
    VAL_RATIO     = 0.15
    TEST_RATIO    = 0.15
    RANDOM_SEED   = 42

    # ─── 증강 ───────────────────────────────────────────────────────────────
    AUG_NOISE_STD    = 0.02         # Gaussian noise
    AUG_SCALE_RANGE  = (0.75, 1.25) # EAR scale jitter (더 넓게)
    AUG_TIME_SHIFT   = 10           # 최대 ±10프레임 shift
    AUG_COPIES       = 8            # drowsy 증강 배수 (원본 대비)

    # ─── 모델 공통 ──────────────────────────────────────────────────────────
    BATCH_SIZE    = 64
    EPOCHS        = 60
    LR            = 5e-4  # LR 낮춰서 안정적 수렴
    WEIGHT_DECAY  = 1e-3  # regularization 강화
    PATIENCE      = 15
    DROPOUT       = 0.5   # 오버피팅 심해서 올림

    # ─── 클래스 불균형 보정 ─────────────────────────────────────────
    DROWSY_WEIGHT    = 3.0   # drowsy 클래스 가중치 (수동 설정)
    FOCAL_GAMMA      = 2.0   # Focal Loss gamma (어려운 샘플 집중)
    PRED_THRESHOLD   = 0.35  # drowsy 판정 임계값 (기본 0.5 → 낮춤)

    # ─── CNN+BiLSTM ─────────────────────────────────────────────────────────
    CNN_CHANNELS  = [32, 64]   # 유지
    CNN_KERNEL    = 3
    LSTM_HIDDEN   = 64    # 모델 축소로 오버피팅 억제
    LSTM_LAYERS   = 1     # 레이어 줄여서 단순화

    # ─── Transformer ────────────────────────────────────────────────────────
    TF_D_MODEL    = 32    # Transformer도 축소
    TF_NHEAD      = 4
    TF_LAYERS     = 2
    TF_DIM_FF     = 64

print('CFG 로드 완료')

import os
from datetime import datetime

# 실행 시각 기반 폴더 생성
run_id   = datetime.now().strftime('%Y%m%d_%H%M%S')
SAVE_DIR = os.path.join(CFG.SAVE_BASE, run_id)
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'저장 폴더: {SAVE_DIR}')

## 3. 데이터 로딩 (JSON + CSV)

In [ ]:
import numpy as np

windows  = np.load(CFG.WINDOWS_PATH)             # (2000, 80, 5) float32
labels   = np.load(CFG.LABELS_PATH)              # (2000,) int64
subjects = np.load(CFG.SUBJECTS_PATH)            # (2000,) str

print(f'windows : {windows.shape}  dtype={windows.dtype}')
print(f'labels  : {labels.shape}   alert={(labels==0).sum()}, drowsy={(labels==1).sum()}')
print(f'subjects: {len(set(subjects))}명')


## 4. Random Split (3 seed 분포 확인)

In [ ]:
from sklearn.model_selection import train_test_split

# seed별 분포 미리 확인 (참고용)
for seed in [42, 123, 777]:
    idx = np.arange(len(windows))
    i_tv, i_te = train_test_split(idx, test_size=0.15, stratify=labels, random_state=seed)
    i_tr, i_vl = train_test_split(i_tv, test_size=0.176, stratify=labels[i_tv], random_state=seed)
    print(f'seed={seed}  Train={len(i_tr)}  Val={len(i_vl)}  Test={len(i_te)}'
          f'  (alert/drowsy test: {(labels[i_te]==0).sum()}/{(labels[i_te]==1).sum()})')

## 5. 전처리 (Circular 변환 + 정규화) + 증강

In [ ]:
from sklearn.preprocessing import StandardScaler

# ── Circular 변환: pitch/yaw/roll → sin/cos ───────────────────────────────
def add_circular_features(X):
    """(N, 80, 5) → (N, 80, 8): ear, mar, sin_p, cos_p, sin_y, cos_y, sin_r, cos_r"""
    rad = np.pi / 180.0
    p = X[:, :, 2:3] * rad
    y = X[:, :, 3:4] * rad
    r = X[:, :, 4:5] * rad
    return np.concatenate([
        X[:, :, :2],
        np.sin(p), np.cos(p),
        np.sin(y), np.cos(y),
        np.sin(r), np.cos(r),
    ], axis=2).astype(np.float32)

# ── 증강 함수 ─────────────────────────────────────────────────────────────
def augment_window(x, rng):
    """단일 윈도우 (80, 8) 증강 — 3가지 기법 랜덤 조합"""
    x = x.copy()

    # 1) Gaussian noise
    if rng.random() < 0.8:
        x += rng.normal(0, CFG.AUG_NOISE_STD, x.shape).astype(np.float32)

    # 2) EAR scale jitter (눈 크기 개인차 시뮬)
    if rng.random() < 0.7:
        scale = rng.uniform(*CFG.AUG_SCALE_RANGE)
        x[:, 0] *= scale  # ear 채널만

    # 3) Time shift (최대 ±AUG_TIME_SHIFT 프레임)
    if rng.random() < 0.6:
        shift = rng.randint(-CFG.AUG_TIME_SHIFT, CFG.AUG_TIME_SHIFT + 1)
        x = np.roll(x, shift, axis=0)

    return x

def augment_aggressive(X, y, seed=CFG.RANDOM_SEED):
    """
    적극적 증강:
    - drowsy: AUG_COPIES배 복제 증강
    - alert:  2배 증강 (다양성 확보)
    """
    rng = np.random.RandomState(seed)
    aug_X, aug_y = [X], [y]  # 원본 포함

    drowsy_idx = np.where(y == 1)[0]
    alert_idx  = np.where(y == 0)[0]

    # drowsy: AUG_COPIES배 증강 (부족한 클래스 집중 보강)
    for _ in range(CFG.AUG_COPIES):
        batch = np.array([augment_window(X[i], rng) for i in drowsy_idx])
        aug_X.append(batch)
        aug_y.append(np.ones(len(drowsy_idx), dtype=np.int64))

    # alert: 2배 증강
    for _ in range(2):
        batch = np.array([augment_window(X[i], rng) for i in alert_idx])
        aug_X.append(batch)
        aug_y.append(np.zeros(len(alert_idx), dtype=np.int64))

    X_out = np.concatenate(aug_X, axis=0)
    y_out = np.concatenate(aug_y, axis=0)
    perm  = rng.permutation(len(y_out))
    return X_out[perm], y_out[perm]

def prepare_data(seed):
    """seed로 split → 전처리 → 반환"""
    idx = np.arange(len(windows))
    i_tv, i_te = train_test_split(idx, test_size=0.15, stratify=labels, random_state=seed)
    i_tr, i_vl = train_test_split(i_tv, test_size=0.176, stratify=labels[i_tv], random_state=seed)

    X_tr_raw = windows[i_tr]; y_tr_raw = labels[i_tr]
    X_vl_raw = windows[i_vl]; y_vl     = labels[i_vl].astype(np.int64)
    X_te_raw = windows[i_te]; y_te     = labels[i_te].astype(np.int64)

    X_tr_aug, y_tr_aug = augment_aggressive(add_circular_features(X_tr_raw), y_tr_raw)
    X_vl_circ = add_circular_features(X_vl_raw)
    X_te_circ = add_circular_features(X_te_raw)

    N, T, F = X_tr_aug.shape
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr_aug.reshape(-1,F)).reshape(N,T,F).astype(np.float32)
    X_vl = sc.transform(X_vl_circ.reshape(-1,F)).reshape(-1,T,F).astype(np.float32)
    X_te = sc.transform(X_te_circ.reshape(-1,F)).reshape(-1,T,F).astype(np.float32)
    X_te_raw_out = X_te_raw.copy()

    return (X_tr, y_tr_aug.astype(np.int64),
            X_vl, y_vl, X_te, y_te, sc, i_te, X_te_raw_out)

print('전처리 함수 정의 완료')


## 6. DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class DrowsinessDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.transpose(0, 2, 1), dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def make_weighted_sampler(y):
    counts  = np.bincount(y, minlength=2)
    weights = 1.0 / (counts + 1e-6)
    return WeightedRandomSampler(torch.tensor(weights[y], dtype=torch.float32),
                                 num_samples=len(y), replacement=True)

def make_loaders(X_tr, y_tr, X_vl, y_vl, X_te, y_te):
    tr = DataLoader(DrowsinessDataset(X_tr, y_tr), batch_size=CFG.BATCH_SIZE,
                    sampler=make_weighted_sampler(y_tr))
    vl = DataLoader(DrowsinessDataset(X_vl, y_vl), batch_size=CFG.BATCH_SIZE, shuffle=False)
    te = DataLoader(DrowsinessDataset(X_te, y_te), batch_size=CFG.BATCH_SIZE, shuffle=False)
    return tr, vl, te

print('DataLoader 함수 정의 완료')


## 7. 모델 정의

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Focal Loss — 쉬운 샘플(alert) 패널티 줄이고 어려운 샘플(drowsy) 집중.
    gamma=0: 일반 CrossEntropy,  gamma↑: 어려운 샘플에 더 집중
    """
    def __init__(self, gamma=CFG.FOCAL_GAMMA, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt  = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

print(f'FocalLoss 정의 완료 (gamma={CFG.FOCAL_GAMMA})')


In [ ]:
import torch.nn as nn
import math

# ── Model 1: CNN + BiLSTM ─────────────────────────────────────────────────
class CNNBiLSTM(nn.Module):
    """
    v1과 동일 구조.
    입력: (B, 8, 80)  출력: (B, 2)
    """
    def __init__(self, n_features=CFG.N_FEATURES):
        super().__init__()
        cnn, in_ch = [], n_features
        for out_ch in CFG.CNN_CHANNELS:
            cnn += [nn.Conv1d(in_ch, out_ch, CFG.CNN_KERNEL, padding=CFG.CNN_KERNEL//2),
                    nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True), nn.MaxPool1d(2, 2)]
            in_ch = out_ch
        self.cnn  = nn.Sequential(*cnn)
        self.lstm = nn.LSTM(CFG.CNN_CHANNELS[-1], CFG.LSTM_HIDDEN, CFG.LSTM_LAYERS,
                            batch_first=True, bidirectional=True,
                            dropout=CFG.DROPOUT if CFG.LSTM_LAYERS > 1 else 0.0)
        self.drop = nn.Dropout(CFG.DROPOUT)
        self.fc   = nn.Linear(CFG.LSTM_HIDDEN * 2, CFG.N_CLASSES)

    def forward(self, x):
        x = self.cnn(x)             # (B, 64, 20)
        x = x.permute(0, 2, 1)     # (B, 20, 64)
        out, _ = self.lstm(x)       # (B, 20, 256)
        return self.fc(self.drop(out[:, -1, :]))


# ── Model 2: CNN + LSTM (단방향) ────────────────────────────────────────
class CNNLSTM(nn.Module):
    """
    단방향 LSTM — 시간 순서(과거→현재)만 학습.
    졸음은 서서히 감기는 패턴이라 역방향 불필요.
    입력: (B, 8, 80)  출력: (B, 2)
    """
    def __init__(self, n_features=CFG.N_FEATURES):
        super().__init__()
        cnn, in_ch = [], n_features
        for out_ch in CFG.CNN_CHANNELS:
            cnn += [nn.Conv1d(in_ch,out_ch,CFG.CNN_KERNEL,padding=CFG.CNN_KERNEL//2),
                    nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True), nn.MaxPool1d(2,2)]
            in_ch = out_ch
        self.cnn  = nn.Sequential(*cnn)
        self.lstm = nn.LSTM(CFG.CNN_CHANNELS[-1], CFG.LSTM_HIDDEN, CFG.LSTM_LAYERS,
                            batch_first=True, bidirectional=False,
                            dropout=CFG.DROPOUT if CFG.LSTM_LAYERS > 1 else 0.0)
        self.drop = nn.Dropout(CFG.DROPOUT)
        self.fc   = nn.Linear(CFG.LSTM_HIDDEN, CFG.N_CLASSES)  # 단방향이라 *1

    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        return self.fc(self.drop(out[:, -1, :]))


# ── Model 3: Temporal Transformer ────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=80, dropout=0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):  # x: (B, T, d_model)
        return self.drop(x + self.pe[:, :x.size(1)])


class TemporalTransformer(nn.Module):
    """
    Transformer 기반 시계열 분류기.
    EAR의 시간적 패턴을 attention으로 학습.
    입력: (B, 8, 80)  출력: (B, 2)
    """
    def __init__(self, n_features=CFG.N_FEATURES):
        super().__init__()
        self.input_proj = nn.Linear(n_features, CFG.TF_D_MODEL)
        self.pos_enc    = PositionalEncoding(CFG.TF_D_MODEL, max_len=CFG.WINDOW_SIZE)
        encoder_layer   = nn.TransformerEncoderLayer(
            d_model=CFG.TF_D_MODEL, nhead=CFG.TF_NHEAD,
            dim_feedforward=CFG.TF_DIM_FF, dropout=CFG.DROPOUT,
            batch_first=True
        )
        self.encoder    = nn.TransformerEncoder(encoder_layer, num_layers=CFG.TF_LAYERS)
        self.drop       = nn.Dropout(CFG.DROPOUT)
        self.fc         = nn.Linear(CFG.TF_D_MODEL, CFG.N_CLASSES)

    def forward(self, x):          # x: (B, 8, 80)
        x = x.permute(0, 2, 1)    # (B, 80, 8)
        x = self.input_proj(x)    # (B, 80, d_model)
        x = self.pos_enc(x)       # (B, 80, d_model)
        x = self.encoder(x)       # (B, 80, d_model)
        x = x.mean(dim=1)         # (B, d_model) — global average pooling
        return self.fc(self.drop(x))


# shape 확인
with torch.no_grad():
    dummy = torch.randn(4, CFG.N_FEATURES, CFG.WINDOW_SIZE)
    m1 = CNNBiLSTM()
    m2 = TemporalTransformer()
    print(f'CNNBiLSTM      출력: {m1(dummy).shape}  파라미터: {sum(p.numel() for p in m1.parameters()):,}')
    print(f'Transformer    출력: {m2(dummy).shape}  파라미터: {sum(p.numel() for p in m2.parameters()):,}')

## 8. 학습 함수

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
try:
    from torch.optim import RAdam
except ImportError:
    RAdam = AdamW  # fallback
from sklearn.metrics import f1_score as sk_f1
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def train_model(model, name, train_loader, val_loader, save_path=None):
    weights   = torch.tensor([1.0, CFG.DROWSY_WEIGHT], dtype=torch.float32).to(DEVICE)
    criterion = FocalLoss(gamma=CFG.FOCAL_GAMMA, weight=weights)
    optimizer = RAdam(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    best_f1, patience_cnt = 0.0, 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

    print(f'\n=== {name} 학습 시작 ===')
    for epoch in range(1, CFG.EPOCHS + 1):
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(Xb)
            loss   = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss    += loss.item() * len(yb)
            tr_correct += (logits.argmax(1) == yb).sum().item()
            tr_total   += len(yb)

        model.eval()
        vl_loss, vl_preds, vl_labels = 0.0, [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                logits  = model(Xb)
                vl_loss += criterion(logits, yb).item() * len(yb)
                vl_preds.extend(logits.argmax(1).cpu().numpy())
                vl_labels.extend(yb.cpu().numpy())

        vl_acc = sum(p == l for p, l in zip(vl_preds, vl_labels)) / len(vl_labels)
        vl_f1  = sk_f1(vl_labels, vl_preds, average='macro', zero_division=0)
        scheduler.step(vl_f1)

        history['train_loss'].append(tr_loss / tr_total)
        history['val_loss'].append(vl_loss / len(vl_labels))
        history['val_acc'].append(vl_acc)
        history['val_f1'].append(vl_f1)

        if epoch % 10 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}/{CFG.EPOCHS} | '
                  f'Train Loss {tr_loss/tr_total:.4f} | '
                  f'Val Loss {vl_loss/len(vl_labels):.4f} Acc {vl_acc:.3f} F1 {vl_f1:.3f}')

        if vl_f1 > best_f1:
            best_f1, patience_cnt = vl_f1, 0
            if save_path:
                torch.save(model.state_dict(), save_path)
        else:
            patience_cnt += 1
            if patience_cnt >= CFG.PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    print(f'  최고 Val F1: {best_f1:.4f}')
    return history


@torch.no_grad()
def get_preds(model, loader, threshold=0.5):
    model.eval()
    preds, trues, probs = [], [], []
    for Xb, yb in loader:
        logits = model(Xb.to(DEVICE))
        p = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds.extend((p >= threshold).astype(int))
        trues.extend(yb.numpy())
        probs.extend(p)
    return np.array(preds), np.array(trues), np.array(probs)


def find_best_threshold(model, loader):
    _, trues, probs = get_preds(model, loader, threshold=0.0)
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.2, 0.70, 0.02):
        preds = (probs >= t).astype(int)
        f1 = sk_f1(trues, preds, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    print(f'  최적 threshold: {best_t:.2f}  (val macro F1={best_f1:.4f})')
    return best_t


print('학습/평가 함수 정의 완료')

## 9. 학습 실행 (CNN+LSTM × 3 seed → 최고 Test F1 선택)

In [ ]:
import random

N_RUNS = 5
SEEDS  = [random.randint(0, 9999) for _ in range(N_RUNS)]
print(f'이번 실행 seeds: {SEEDS}')

seed_results = []

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'Seed {seed} 학습 시작')
    print('='*60)

    # 데이터 준비
    X_tr, y_tr, X_vl, y_vl, X_te, y_te, scaler_s, i_te, X_te_raw = prepare_data(seed)
    tr_loader, vl_loader, te_loader = make_loaders(X_tr, y_tr, X_vl, y_vl, X_te, y_te)

    # 모델 초기화 & 학습
    model_s   = CNNLSTM().to(DEVICE)
    save_path = os.path.join(SAVE_DIR, f'cnn_lstm_seed{seed}.pt')
    hist      = train_model(model_s, f'CNN+LSTM (seed={seed})', tr_loader, vl_loader, save_path=save_path)

    # best weight 로드 후 threshold 탐색
    model_s.load_state_dict(torch.load(save_path, map_location=DEVICE))
    best_t = find_best_threshold(model_s, vl_loader)

    # Test 평가
    preds, trues, probs = get_preds(model_s, te_loader, threshold=best_t)
    f1 = sk_f1(trues, preds, average='macro', zero_division=0)

    seed_results.append({
        'seed':      seed,
        'model':     model_s,
        'scaler':    scaler_s,
        'threshold': best_t,
        'test_f1':   f1,
        'i_te':      i_te,
        'X_te_raw':  X_te_raw,
        'y_te':      y_te,
        'te_loader': te_loader,
        'history':   hist,
        'save_path': save_path,
    })
    print(f'Seed {seed} → Test F1: {f1:.4f}  threshold: {best_t:.2f}')

# 최고 Test F1 seed 선택
best = max(seed_results, key=lambda x: x['test_f1'])
print(f'\n{"="*60}')
print(f'최고 Test F1: seed={best["seed"]}  F1={best["test_f1"]:.4f}')
print('='*60)

for r in seed_results:
    mark = ' ← best' if r['seed'] == best['seed'] else ''
    print(f'  seed={r["seed"]}  Test F1={r["test_f1"]:.4f}{mark}')

# 전역 변수로 노출 (이하 셀에서 사용)
best_model = best['model']
scaler     = best['scaler']
threshold  = best['threshold']
X_test_raw = best['X_te_raw']
y_test     = best['y_te']
test_idx   = best['i_te']
te_loader  = best['te_loader']
best_hist  = best['history']

# 빈 셀 (제거용)
pass

In [ ]:
# 학습 곡선 (best seed)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'학습 곡선 (seed={best["seed"]}, Test F1={best["test_f1"]:.4f})', fontsize=13)

h = best_hist
epochs = range(1, len(h['train_loss']) + 1)

axes[0].plot(epochs, h['train_loss'], label='Train Loss')
axes[0].plot(epochs, h['val_loss'],   label='Val Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs, h['val_f1'], label='Val F1', color='green')
axes[1].set_title('Val Macro F1'); axes[1].legend(); axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

In [ ]:
# 빈 셀 (제거용)
pass

In [ ]:
# 빈 셀 (제거용)
pass

## 10. 평가 (Rule-based vs CNN+LSTM)

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, roc_auc_score)

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CFG.CLASS_NAMES, yticklabels=CFG.CLASS_NAMES)
    plt.title(title); plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout(); plt.show()


# ── Rule-based (subject 평균 EAR 기반 개인화 PERCLOS) ────────────────────
subjects_all = np.load(CFG.SUBJECTS_PATH)  # (2000,)

subject_alert_ear = {}
for subj in set(subjects_all):
    subj_mask   = subjects_all == subj
    alert_mask  = labels == 0
    alert_clips = windows[subj_mask & alert_mask]
    if len(alert_clips) > 0:
        subject_alert_ear[subj] = float(alert_clips[:, :, 0].mean())
    else:
        subject_alert_ear[subj] = 0.25

def rule_based_pred(X_raw, subj_ids, perclos_thresh=0.3):
    ear      = X_raw[:, :, 0]
    ref      = np.array([subject_alert_ear.get(s, 0.25) for s in subj_ids])
    ear_norm = ear / (ref[:, None] + 1e-6)
    perclos  = np.mean(ear_norm < 0.8, axis=1)
    return (perclos >= perclos_thresh).astype(int)

test_subjects = subjects_all[test_idx]
rb_preds = rule_based_pred(X_test_raw, test_subjects)

# ── CNN+LSTM (best seed) 추론 ─────────────────────────────────────────────
dl_preds, y_true, dl_probs = get_preds(best_model, te_loader, threshold=threshold)

# ── 비교표 ────────────────────────────────────────────────────────────────
results = []
for name, preds, probs, true in [
    ('Rule-based (개인화 PERCLOS)', rb_preds,  None,     y_test),
    ('CNN+LSTM (단방향)',           dl_preds,  dl_probs, y_true),
]:
    acc = accuracy_score(true, preds)
    f1m = f1_score(true, preds, average='macro',  zero_division=0)
    f1d = f1_score(true, preds, pos_label=1,      zero_division=0)
    auc = roc_auc_score(true, probs) if probs is not None else float('nan')
    results.append({'Model': name, 'Acc': f'{acc:.4f}',
                    'F1(macro)': f'{f1m:.4f}',
                    'F1(drowsy)': f'{f1d:.4f}',
                    'AUC': f'{auc:.4f}'})
    print(f'\n=== {name} ===')
    print(classification_report(true, preds, target_names=CFG.CLASS_NAMES, zero_division=0))
    plot_cm(true, preds, name)

print('\n=== 최종 비교 ===')
print(pd.DataFrame(results).to_string(index=False))

In [ ]:
# 빈 셀 (제거용)
pass

## 11. 최종 모델 저장

In [ ]:
# best_model_v2.pt 저장 (inference.py에서 그대로 로드 가능)
checkpoint = {
    'model_state': best_model.state_dict(),
    'scaler':      scaler,
    'threshold':   threshold,
    'model_name':  'CNN+LSTM (단방향)',
    'cfg': {
        'n_features':    CFG.N_FEATURES,
        'window_size':   CFG.WINDOW_SIZE,
        'class_names':   CFG.CLASS_NAMES,
        'feature_names': ['ear', 'mar', 'sin_p', 'cos_p', 'sin_y', 'cos_y', 'sin_r', 'cos_r'],
    }
}

save_path = os.path.join(SAVE_DIR, 'best_model_v2.pt')
torch.save(checkpoint, save_path)

# 루트 경로에도 복사 (편의용)
import shutil
root_copy = '/content/drive/MyDrive/internship/best_model_v2.pt'
shutil.copy(save_path, root_copy)

print(f'저장 완료:')
print(f'  {save_path}')
print(f'  {root_copy}')
print(f'  threshold: {threshold:.2f}  |  seed: {best["seed"]}  |  Test F1: {best["test_f1"]:.4f}')

# 빈 셀 (제거용)
pass

In [ ]:
# 빈 셀 (제거용)
pass